In [1]:
# 1 模型训练SA + 训练增强完整版
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import xarray as xr
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
from tqdm import tqdm

# ===========================
# 路径配置
# ===========================
DATA_PATH = "/mnt/g/次季节模型/202601训练/温度/数据/D.nc"
BASE_DIR = "/mnt/g/次季节模型/202601训练/温度/模型/SSA"
os.makedirs(BASE_DIR, exist_ok=True)

METRIC_CSV = os.path.join(BASE_DIR, "metrics.csv")
CKPT_PATH  = os.path.join(BASE_DIR, "checkpoint.pth")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 1
EPOCHS = 530
LR = 1e-3
MAX_GRAD_NORM = 1.0

# ===========================
# 变量
# ===========================
input_vars = [f'tas_hist_{i}' for i in range(20)] + \
             [f'gh200_hist_{i}' for i in range(10)] + \
             [f'gh200_300_hist_{i}' for i in range(10)] + \
             [f'gh200_500_hist_{i}' for i in range(10)] + \
             ['pred_t2m_month', 'elevation']

target_vars = [
    'tas_mon','tas_wed','tas_fri',
    'tas_sun','tas_tue_next','tas_thu_next','tas_sat_next'
]

# ===========================
# 数据加载
# ===========================
ds = xr.open_dataset(DATA_PATH)
for v in input_vars + target_vars:
    ds[v] = ds[v].fillna(0)

# ===========================
# Dataset
# ===========================
class SubseasonalDataset(Dataset):
    def __init__(self, ds):
        self.x = ds[input_vars].to_array().transpose(
            'time','variable','latitude','longitude'
        )
        self.y = ds[target_vars].to_array().transpose(
            'time','variable','latitude','longitude'
        )

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return (
            torch.tensor(self.x[idx].values, dtype=torch.float32),
            torch.tensor(self.y[idx].values, dtype=torch.float32)
        )

dataset = SubseasonalDataset(ds)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ===========================
# 模型
# ===========================
# ===========================
# 高性能 Attention 模块
# ===========================
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)


# ===========================
# Res + 双注意力 + 空洞卷积
# ===========================
class AdvancedResidualBlock(nn.Module):
    def __init__(self, cin, cout, dilation=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, padding=dilation, dilation=dilation)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=dilation, dilation=dilation)

        self.sa = SpatialAttention()
        self.ca = ChannelAttention(cout)

        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        y = self.relu(self.c1(x))
        y = self.c2(y)
        y = self.ca(self.sa(y))
        return self.relu(y + self.skip(x))


# ===========================
# U-Net Encoder
# ===========================
class EncoderBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            AdvancedResidualBlock(cin, cout, dilation=1),
            AdvancedResidualBlock(cout, cout, dilation=2),
        )
        self.pool = nn.AvgPool2d(2)

    def forward(self, x):
        f = self.block(x)
        return self.pool(f), f


# ===========================
# U-Net Decoder（自动对齐奇偶尺寸）
# ===========================
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.block = nn.Sequential(
            AdvancedResidualBlock(out_channels + skip_channels, out_channels),
            AdvancedResidualBlock(out_channels, out_channels),
        )

    def forward(self, x, skip):
        x = self.up(x)
        # 对齐 skip 特征尺寸
        if x.size()[2:] != skip.size()[2:]:
            x = F.interpolate(x, size=skip.size()[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.block(x)


# ===========================
# CPU 顶配复杂模型
# ===========================
class AdvancedUNetResNet(nn.Module):
    def __init__(self, input_channels=len(input_vars), output_channels=len(target_vars), base=32):
        super().__init__()

        # ================= Stem =================
        self.stem = nn.Conv2d(input_channels, base, 3, padding=1)

        # ================= Encoder =================
        self.e1 = EncoderBlock(base, base*2)     # 32 → 64
        self.e2 = EncoderBlock(base*2, base*4)   # 64 → 128

        # ================= Bottleneck =================
        self.bottleneck = nn.Sequential(
            AdvancedResidualBlock(base*4, base*4, dilation=2),
            AdvancedResidualBlock(base*4, base*4, dilation=4),
        )

        # ================= Decoder =================
        self.d2 = DecoderBlock(in_channels=base*4, skip_channels=base*4, out_channels=base*2)
        self.d1 = DecoderBlock(in_channels=base*2, skip_channels=base*2, out_channels=base)

        # ================= Head =================
        self.head = nn.Sequential(
            nn.Conv2d(base, base, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(base, output_channels, 1)
        )

    def forward(self, x):
        x = self.stem(x)
        x1, skip1 = self.e1(x)
        x2, skip2 = self.e2(x1)
        x = self.bottleneck(x2)
        x = self.d2(x, skip2)
        x = self.d1(x, skip1)
        return self.head(x)


# ===========================
# 模型实例化
# ===========================
model = AdvancedUNetResNet().to(DEVICE)

# ===========================
# EMA
# ===========================
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.ema = self._clone(model)
        self.decay = decay
        self.ema.eval()

    def _clone(self, model):
        ema = type(model)()
        ema.load_state_dict(model.state_dict())
        ema.to(next(model.parameters()).device)
        for p in ema.parameters():
            p.requires_grad_(False)
        return ema

    @torch.no_grad()
    def update(self, model):
        for p_ema, p in zip(self.ema.parameters(), model.parameters()):
            p_ema.data.mul_(self.decay).add_(p.data * (1 - self.decay))

    def state_dict(self):
        return self.ema.state_dict()

    def load_state_dict(self, sd):
        self.ema.load_state_dict(sd)

ema = ModelEMA(model, decay=0.999)

# ===========================
# 优化器 + AMP + Scheduler
# ===========================
optimizer = optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

scaler = torch.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10, verbose=True
)

# ===========================
# 指标函数（纯 PyTorch）
# ===========================
def compute_rmse(pred, truth):
    return torch.sqrt(torch.mean((pred - truth) ** 2)).item()

def compute_nac(pred, truth):
    numerator = torch.sum((pred - truth) ** 2)
    denominator = torch.sum((truth - torch.mean(truth)) ** 2)
    return (1 - numerator / denominator).item() if denominator > 0 else 0.0

def compute_acc_anomaly(pred, truth):
    y_true = truth.view(-1)
    y_pred = pred.view(-1)
    y_true_anom = y_true - y_true.mean()
    y_pred_anom = y_pred - y_pred.mean()
    num = torch.sum(y_true_anom * y_pred_anom)
    den = torch.sqrt(torch.sum(y_true_anom**2) * torch.sum(y_pred_anom**2) + 1e-8)
    return (num / den).item()

def compute_crps(pred, truth):
    return torch.mean(torch.abs(pred - truth)).item()

# ===========================
# 断点续训
# ===========================
start_epoch = 0
best = dict(RMSE=1e9, ACC=-1e9, NAC=-1e9, CRPS=1e9)

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    ema.load_state_dict(ckpt["ema"])
    optimizer.load_state_dict(ckpt["optim"])
    best = ckpt["best"]
    start_epoch = ckpt["epoch"] + 1
    print(f"✅ 从 epoch {start_epoch} 续训")

# ===========================
# CSV 初始化
# ===========================
if not os.path.exists(METRIC_CSV):
    pd.DataFrame(columns=["epoch","RMSE","ACC","NAC","CRPS"]).to_csv(METRIC_CSV,index=False)

# ===========================
# 训练循环
# ===========================
for epoch in range(start_epoch, EPOCHS):

    model.train()
    losses = []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=120)

    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()

        
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type=="cuda")):
                pred = model(x)
                loss = loss_fn(pred, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()

        ema.update(model)

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{np.mean(losses):.4f}")

    train_loss = np.mean(losses)

    # ===== EMA 评估 =====
    model_eval = ema.ema
    model_eval.eval()

    rmses, accs, nacs, crpss = [], [], [], []

    with torch.no_grad():
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            p = model_eval(x)
            rmses.append(compute_rmse(p, y))
            accs.append(compute_acc_anomaly(p, y))
            nacs.append(compute_nac(p, y))
            crpss.append(compute_crps(p, y))

    RM, AC, NA, CR = map(np.mean, [rmses, accs, nacs, crpss])
    scheduler.step(RM)

    # ===== CSV 写入 =====
    pd.DataFrame(
        [[epoch, RM, AC, NA, CR]],
        columns=["epoch", "RMSE", "ACC", "NAC", "CRPS"]
    ).to_csv(METRIC_CSV, mode="a", header=False, index=False)

    # ================= 保存最优 EMA ==================
    star_rmse = RM < best["RMSE"]
    star_acc  = AC > best["ACC"]
    star_nac  = NA > best["NAC"]
    star_crps = CR < best["CRPS"]

    saved_models = []

    if star_rmse:
        best["RMSE"] = RM
        fname = "SSAttentionResNet_tas_best_RMSE.pth"
        torch.save(ema.state_dict(), os.path.join(BASE_DIR, fname))
        saved_models.append(fname)

    if star_acc:
        best["ACC"] = AC
        fname = "SSAttentionResNet_tas_best_ACC.pth"
        torch.save(ema.state_dict(), os.path.join(BASE_DIR, fname))
        saved_models.append(fname)

    if star_nac:
        best["NAC"] = NA
        fname = "SSAttentionResNet_tas_best_NAC.pth"
        torch.save(ema.state_dict(), os.path.join(BASE_DIR, fname))
        saved_models.append(fname)

    if star_crps:
        best["CRPS"] = CR
        fname = "SSAttentionResNet_tas_best_CRPS.pth"
        torch.save(ema.state_dict(), os.path.join(BASE_DIR, fname))
        saved_models.append(fname)

    # ===== checkpoint =====
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "ema": ema.state_dict(),
        "optim": optimizer.state_dict(),
        "best": best
    }, CKPT_PATH)

    # ===== 日志打印 =====
    print(
        f"\nEpoch {epoch+1:03d} | "
        f"Loss={train_loss:.4f} | "
        f"RMSE={RM:.4f} | ACC={AC:.4f} | NAC={NA:.4f} | CRPS={CR:.4f} | "
        f"LR={optimizer.param_groups[0]['lr']:.2e}"
    )

    # ⭐ 标记
    star_list = []
    if star_rmse: star_list.append("RMSE")
    if star_acc:  star_list.append("ACC")
    if star_nac:  star_list.append("NAC")
    if star_crps: star_list.append("CRPS")
    star_info = " ".join([f"⭐{s}" for s in star_list]) if star_list else "-"

    # 简洁日志
    print(
        f"Epoch {epoch+1:03d} | "
        f"loss={train_loss:.4f} | "
        f"RMSE={RM:.4f}{'⭐' if star_rmse else ''} | "
        f"ACC={AC:.4f}{'⭐' if star_acc else ''} | "
        f"NAC={NA:.4f}{'⭐' if star_nac else ''} | "
        f"CRPS={CR:.4f}{'⭐' if star_crps else ''}"
    )

    print(f"Save: {star_info} | CSV ✔ | CKPT ✔\n")


/home/iii/anaconda3/envs/NN/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/tmp/ipykernel_494/379831784.py:288: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have

✅ 从 epoch 520 续训


Epoch 521/530: 100%|█████████████████████████████████████| 311/311 [01:59<00:00,  2.61it/s, avg=2490.2715, loss=25.4599]



Epoch 521 | Loss=2490.2715 | RMSE=13.2196 | ACC=0.9478 | NAC=0.9244 | CRPS=11.8832 | LR=6.25e-05
Epoch 521 | loss=2490.2715 | RMSE=13.2196⭐ | ACC=0.9478⭐ | NAC=0.9244⭐ | CRPS=11.8832⭐
Save: ⭐RMSE ⭐ACC ⭐NAC ⭐CRPS | CSV ✔ | CKPT ✔



Epoch 522/530: 100%|█████████████████████████████████████| 311/311 [01:50<00:00,  2.82it/s, avg=2501.3505, loss=36.5020]


...


Epoch 528 | Loss=2495.5914 | RMSE=13.2135 | ACC=0.9478 | NAC=0.9245 | CRPS=11.8779 | LR=6.25e-05
Epoch 528 | loss=2495.5914 | RMSE=13.2135⭐ | ACC=0.9478⭐ | NAC=0.9245⭐ | CRPS=11.8779⭐
Save: ⭐RMSE ⭐ACC ⭐NAC ⭐CRPS | CSV ✔ | CKPT ✔



Epoch 529/530: 100%|█████████████████████████████████████| 311/311 [01:50<00:00,  2.82it/s, avg=2502.3362, loss=18.8140]



Epoch 529 | Loss=2502.3362 | RMSE=13.2124 | ACC=0.9479 | NAC=0.9246 | CRPS=11.8767 | LR=6.25e-05
Epoch 529 | loss=2502.3362 | RMSE=13.2124⭐ | ACC=0.9479⭐ | NAC=0.9246⭐ | CRPS=11.8767⭐
Save: ⭐RMSE ⭐ACC ⭐NAC ⭐CRPS | CSV ✔ | CKPT ✔



Epoch 530/530: 100%|█████████████████████████████████████| 311/311 [01:49<00:00,  2.83it/s, avg=2507.1228, loss=21.1733]



Epoch 530 | Loss=2507.1228 | RMSE=13.2105 | ACC=0.9479 | NAC=0.9246 | CRPS=11.8750 | LR=6.25e-05
Epoch 530 | loss=2507.1228 | RMSE=13.2105⭐ | ACC=0.9479⭐ | NAC=0.9246⭐ | CRPS=11.8750⭐
Save: ⭐RMSE ⭐ACC ⭐NAC ⭐CRPS | CSV ✔ | CKPT ✔

